# Эксперимент: сравнение моделей GigaChat и DeepSeek

Каждой модели отправляем 4 промпта по 5 раз с одинаковыми параметрами генерации.
Для каждого запроса фиксируем время ответа, число токенов и условную стоимость.

- Ответы моделей сохраняются в `results/<модель>/<задача>/run_<k>.md` (метаданные в YAML-заголовке файла).
- Метрики всех запросов — в `results/runs.csv`, сводка — в `results/summary.csv`.
- Если прогон прервался, ячейку можно перезапустить: уже сохранённые ответы пропускаются.

In [ ]:
%pip install gigachat openai python-dotenv pandas

## Параметры

In [1]:
import csv
import os
import time
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from gigachat import GigaChat
from gigachat.exceptions import AuthenticationError, RateLimitError, ServerError
from gigachat.models import ChatCompletionRequest, ChatMessage, ChatModelOptions
from openai import OpenAI

load_dotenv()

RUNS = 5              # прогонов на каждый промпт
TEMPERATURE = 0.7     # одинаковая для всех моделей
MAX_TOKENS = 8192     # с запасом, чтобы длинные ответы не обрезались
RESULTS = Path("results")
CSV_PATH = RESULTS / "runs.csv"

GIGA_MODELS = ["GigaChat-3-Pro", "GigaChat-3-Lightning"]
DS_MODELS = ["deepseek-v4-pro", "deepseek-v4.1-flash"]

# Цена, ₽ за 1M токенов (вход и выход тарифицируются одинаково)
DS_PRICE = 92.11 / 5  # 92,11 ₽ за 5M токенов у агрегатора, одна цена для всех моделей DeepSeek
PRICE_PER_1M = {
    "GigaChat-3-Pro": 1500 / 3,         # пакет GigaChat 2 Pro: 1500 ₽ за 3M
    "GigaChat-3-Lightning": 1300 / 20,  # пакет GigaChat 2 Lite: 1300 ₽ за 20M
    "deepseek-v4-pro": DS_PRICE,
    "deepseek-v4.1-flash": DS_PRICE,
}
PRICE_PER_1M

{'GigaChat-3-Pro': 500.0,
 'GigaChat-3-Lightning': 65.0,
 'deepseek-v4-pro': 18.422,
 'deepseek-v4.1-flash': 18.422}

## Промпты

In [2]:
prompts = [
    {
        "name": "python",
        "prompt": (
            "Напиши функцию на Python, которая принимает строку и возвращает словарь, "
            "где ключи — слова, а значения — сколько раз каждое слово встречается в строке. "
            "Регистр не учитывать, знаки препинания игнорировать. "
            "Приведи код, кратко объясни решение и покажи пример вызова с результатом."
        ),
    },
    {
        "name": "c",
        "prompt": (
            "Напиши программу на C, которая считывает с клавиатуры массив из n целых чисел "
            "(n вводится первым), сортирует его пузырьковой сортировкой и выводит результат. "
            "Память выдели динамически и не забудь её освободить. "
            "Приведи полный компилируемый код и кратко поясни, как он работает."
        ),
    },
    {
        "name": "math",
        "prompt": (
            "Дай строгое определение математического ожидания случайной величины "
            "для дискретного и непрерывного случаев. "
            "Перечисли основные свойства математического ожидания (линейность, ожидание константы, "
            "ожидание произведения независимых величин и т.д.) с краткими доказательствами. "
            "Приведи по одному примеру вычисления для дискретной и непрерывной величины."
        ),
    },
    {
        "name": "literature",
        "prompt": (
            "Опиши главных действующих лиц рассказа И. С. Тургенева «Муму». "
            "Для каждого персонажа укажи: кто он, черты характера, роль в сюжете. "
            "Отдельно объясни, в чём смысл образа Герасима и какую идею хотел донести автор."
        ),
    },
]

## Клиенты

In [3]:
giga = GigaChat(
    credentials=os.getenv("GIGACHAT_AUTH_KEY"),
    scope="GIGACHAT_API_PERS",
    ca_bundle_file=os.path.expanduser("~/certs/russian_trusted_root_ca.crt"),
)
giga.get_token()

ds = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://ai.starimg.ru/v1",
)

## Функции запроса

Обе функции возвращают одинаковый словарь: текст ответа, время, токены, причину остановки.
Время меряем от отправки запроса до получения полного ответа.

In [4]:
def ask_giga(model, prompt, retries=3):
    req = ChatCompletionRequest(
        model=model,
        messages=[ChatMessage(role="user", content=prompt)],
        model_options=ChatModelOptions(temperature=TEMPERATURE, max_tokens=MAX_TOKENS),
    )
    for attempt in range(retries):
        try:
            giga.get_token()  # обновит токен, если он истёк (живёт 30 минут)
            t0 = time.perf_counter()
            resp = giga.chat.create(req)
            latency = time.perf_counter() - t0
            break
        except AuthenticationError:
            giga.get_token()
        except (RateLimitError, ServerError):
            time.sleep(10 * (attempt + 1))
    else:
        raise RuntimeError(f"{model}: не удалось получить ответ за {retries} попытки")

    u = resp.usage
    return {
        "text": resp.messages[0].content[0].text,
        "latency_s": round(latency, 3),
        "input_tokens": u.input_tokens,
        "output_tokens": u.output_tokens,
        "reasoning_tokens": 0,
        "total_tokens": u.total_tokens,
        "finish_reason": resp.finish_reason or resp.messages[0].finish_reason,
    }


def ask_ds(model, prompt, retries=3):
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            resp = ds.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            latency = time.perf_counter() - t0
            break
        except Exception as e:
            print(f"{model}: {e!r}, повтор")
            time.sleep(10 * (attempt + 1))
    else:
        raise RuntimeError(f"{model}: не удалось получить ответ за {retries} попытки")

    u = resp.usage
    details = getattr(u, "completion_tokens_details", None)
    return {
        "text": resp.choices[0].message.content,
        "latency_s": round(latency, 3),
        "input_tokens": u.prompt_tokens,
        "output_tokens": u.completion_tokens,
        "reasoning_tokens": getattr(details, "reasoning_tokens", None) or 0,
        "total_tokens": u.total_tokens,
        "finish_reason": resp.choices[0].finish_reason,
    }


def ask(model, prompt):
    return ask_giga(model, prompt) if model in GIGA_MODELS else ask_ds(model, prompt)

## Проверка перед запуском

Один короткий запрос к каждой модели: проверяем, что параметры принимаются и usage приходит.

Для GigaChat смотрим баланс до и после запроса, чтобы понять, из какого пакета списываются токены
моделей GigaChat-3. Для DeepSeek баланс до и после нужно сверить в личном кабинете агрегатора:
списание должно быть равно `total_tokens × коэффициент`.

In [ ]:
def giga_balance():
    try:
        return {b.usage: b.value for b in giga.get_balance().balance}
    except Exception as e:
        print(f"get_balance недоступен: {e!r}")
        return {}


for model in GIGA_MODELS + DS_MODELS:
    before = giga_balance() if model in GIGA_MODELS else {}
    r = ask(model, "Привет, как дела?")
    print("{model}: {latency_s} с, токены {input_tokens} + {output_tokens} = {total_tokens}, "
          "finish_reason={finish_reason}".format(model=model, **r))
    if before:
        after = giga_balance()
        spent = {k: before[k] - after.get(k, 0) for k in before if before[k] != after.get(k, 0)}
        print(f"    списано с баланса: {spent}")

## Прогон

Порядок: сначала первый прогон всех моделей по всем задачам, потом второй и т.д.
Так случайные задержки сети распределяются между моделями равномерно.

In [ ]:
FIELDS = ["model", "task", "run", "timestamp", "temperature", "max_tokens", "latency_s",
          "input_tokens", "output_tokens", "reasoning_tokens", "total_tokens", "finish_reason", "cost_rub", "file"]


def save_md(path, meta, text):
    header = "\n".join(f"{k}: {v}" for k, v in meta.items())
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(f"---\n{header}\n---\n\n{text}\n", encoding="utf-8")


def append_csv(row):
    new = not CSV_PATH.exists()
    RESULTS.mkdir(exist_ok=True)
    with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)


total = RUNS * len(prompts) * len(GIGA_MODELS + DS_MODELS)
done = 0
for run in range(1, RUNS + 1):
    for model in GIGA_MODELS + DS_MODELS:
        for p in prompts:
            done += 1
            path = RESULTS / model / p["name"] / f"run_{run}.md"
            if path.exists():
                continue
            r = ask(model, p["prompt"])
            meta = {
                "model": model,
                "task": p["name"],
                "run": run,
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "temperature": TEMPERATURE,
                "max_tokens": MAX_TOKENS,
                "latency_s": r["latency_s"],
                "input_tokens": r["input_tokens"],
                "output_tokens": r["output_tokens"],
                "reasoning_tokens": r["reasoning_tokens"],
                "total_tokens": r["total_tokens"],
                "finish_reason": r["finish_reason"],
                "cost_rub": round(r["total_tokens"] * PRICE_PER_1M[model] / 1e6, 5),
            }
            save_md(path, meta, r["text"])
            append_csv({**meta, "file": path.as_posix()})
            print("[{done}/{total}] {model} · {task} · run {run}: {latency_s} с, {total_tokens} ток., {finish_reason}"
                  .format(done=done, total=total, **meta))

## Пересчёт стоимости

Стоимость считается из сохранённых токенов, API не вызывается. Если поменялись цены в `PRICE_PER_1M`,
достаточно запустить ячейку «Параметры», эту ячейку и «Сводку»: обновятся `runs.csv` и `cost_rub` в заголовках md-файлов.

In [5]:
import re

import pandas as pd

df = pd.read_csv(CSV_PATH)
df["cost_rub"] = (df["total_tokens"] * df["model"].map(PRICE_PER_1M) / 1e6).round(5)
df.to_csv(CSV_PATH, index=False)

for row in df.itertuples():
    path = Path(row.file)
    text = path.read_text(encoding="utf-8")
    text = re.sub(r"^cost_rub: .*$", f"cost_rub: {row.cost_rub}", text, count=1, flags=re.M)
    path.write_text(text, encoding="utf-8")

df.groupby("model")["cost_rub"].sum().round(3)

model
GigaChat-3-Lightning    1.001
GigaChat-3-Pro          9.648
deepseek-v4-pro         1.443
deepseek-v4.1-flash     1.359
Name: cost_rub, dtype: float64

## Сводка

In [6]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
truncated = (df["finish_reason"] == "length").sum()
print(f"Запросов: {len(df)}, обрезано по max_tokens: {truncated}")

summary = df.groupby(["task", "model"]).agg(
    runs=("run", "count"),
    latency_mean=("latency_s", "mean"),
    latency_median=("latency_s", "median"),
    latency_min=("latency_s", "min"),
    latency_max=("latency_s", "max"),
    output_tokens_mean=("output_tokens", "mean"),
    output_tokens_median=("output_tokens", "median"),
    total_tokens_mean=("total_tokens", "mean"),
    total_tokens_median=("total_tokens", "median"),
    cost_rub_mean=("cost_rub", "mean"),
    cost_rub_median=("cost_rub", "median"),
)
# стоимость на 1000 запросов: среднее и медиана
summary["cost_per_1000_mean"] = summary["cost_rub_mean"] * 1000
summary["cost_per_1000_median"] = summary["cost_rub_median"] * 1000
summary = summary.drop(columns=["cost_rub_mean", "cost_rub_median"]).round(2)
summary.to_csv(RESULTS / "summary.csv")
summary

Запросов: 80, обрезано по max_tokens: 6


runs  latency_mean  latency_median  \
task       model                                                      
c          GigaChat-3-Lightning     5          2.79            2.39   
           GigaChat-3-Pro           5          5.30            5.20   
           deepseek-v4-pro          5         23.28           22.07   
           deepseek-v4.1-flash      5          8.01            7.34   
literature GigaChat-3-Lightning     5          2.69            2.60   
           GigaChat-3-Pro           5          7.63            7.50   
           deepseek-v4-pro          5        170.49          192.16   
           deepseek-v4.1-flash      5         32.91           31.99   
math       GigaChat-3-Lightning     5          3.44            3.54   
           GigaChat-3-Pro           5         11.39           11.84   
           deepseek-v4-pro          5        403.80          417.06   
           deepseek-v4.1-flash      5         32.47           26.59   
python     GigaChat-3-Lightning     5          1.16            1.13   
           GigaChat-3-Pro           5          3.60            3.54   
           deepseek-v4-pro          5         29.06           24.83   
           deepseek-v4.1-flash      5         18.66           23.38   

                                 latency_min  latency_max  output_tokens_mean  \
task       model                                                                
c          GigaChat-3-Lightning         2.31         4.14               719.4   
           GigaChat-3-Pro               4.99         5.70               704.6   
           deepseek-v4-pro             18.59        28.41              1351.0   
           deepseek-v4.1-flash          6.07        10.81              1534.4   
literature GigaChat-3-Lightning         2.46         3.08               790.8   
           GigaChat-3-Pro               6.30         8.59               938.4   
           deepseek-v4-pro             57.66       291.73              3991.6   
           deepseek-v4.1-flash         24.12        47.41              5396.2   
math       GigaChat-3-Lightning         2.79         3.75              1031.8   
           GigaChat-3-Pro               9.09        14.12              1513.8   
           deepseek-v4-pro            280.36       468.83              8192.0   
           deepseek-v4.1-flash         25.75        51.78              6415.8   
python     GigaChat-3-Lightning         1.05         1.29               311.2   
           GigaChat-3-Pro               3.42         4.00               432.6   
           deepseek-v4-pro             20.72        49.82              1426.0   
           deepseek-v4.1-flash          4.69        36.38              1016.6   

                                 output_tokens_median  total_tokens_mean  \
task       model                                                           
c          GigaChat-3-Lightning                 704.0              785.0   
           GigaChat-3-Pro                       706.0              775.4   
           deepseek-v4-pro                     1523.0             1536.0   
           deepseek-v4.1-flash                 1381.0             1635.2   
literature GigaChat-3-Lightning                 758.0              852.0   
           GigaChat-3-Pro                       937.0             1002.2   
           deepseek-v4-pro                     3048.0             4162.6   
           deepseek-v4.1-flash                 5875.0             5477.6   
math       GigaChat-3-Lightning                1063.0             1068.6   
           GigaChat-3-Pro                      1584.0             1581.4   
           deepseek-v4-pro                     8192.0             8370.0   
           deepseek-v4.1-flash                 6120.0             6512.6   
python     GigaChat-3-Lightning                 301.0              373.4   
           GigaChat-3-Pro                       425.0              500.0   
           deepseek-v4-pro                     1214.0             1598.0   
   

In [7]:
by_model = df.groupby("model").agg(
    requests=("run", "count"),
    latency_mean=("latency_s", "mean"),
    latency_median=("latency_s", "median"),
    total_tokens=("total_tokens", "sum"),
    cost_rub=("cost_rub", "sum"),
    cost_rub_mean=("cost_rub", "mean"),
    cost_rub_median=("cost_rub", "median"),
)
by_model["tokens_per_s"] = df.groupby("model")["output_tokens"].sum() / df.groupby("model")["latency_s"].sum()
by_model["cost_per_1000_mean"] = by_model["cost_rub_mean"] * 1000
by_model["cost_per_1000_median"] = by_model["cost_rub_median"] * 1000
by_model = by_model.drop(columns=["cost_rub_mean", "cost_rub_median"]).round(2)
by_model.to_csv(RESULTS / "by_model.csv")
by_model

,requests,latency_mean,latency_median,total_tokens,cost_rub,tokens_per_s,cost_per_1000_mean,cost_per_1000_median
model,,,,,,,,
GigaChat-3-Lightning,20,2.52,2.56,15395,1.00,283.25,50.03,51.15
GigaChat-3-Pro,20,6.98,6.00,19295,9.65,128.55,482.38,411.75
deepseek-v4-pro,20,156.66,53.74,78333,1.44,23.87,72.15,47.36
deepseek-v4.1-flash,20,23.01,24.94,73777,1.36,156.04,67.96,50.02


## Дополнительный прогон: deepseek-v4-pro, математика без лимита токенов

В основном прогоне deepseek-v4-pro на математике упирался в `max_tokens=8192` во всех 5 прогонах:
почти весь лимит уходил на скрытые рассуждения. Остальные модели в лимит укладываются.
Здесь один запрос deepseek-v4-pro по математике без ограничения длины (`max_tokens` не передаётся),
остальные параметры те же.

Результаты — отдельно: `results_unlimited/deepseek-v4-pro/math/run_1.md` и `results_unlimited/runs.csv`.
Нужны ячейки «Параметры», «Промпты», «Клиенты». Ответ может идти 10–20 минут.

In [8]:
import pandas as pd

UNLIMITED = Path("results_unlimited")
UNL_CSV = UNLIMITED / "runs.csv"
MODEL = "deepseek-v4-pro"
math_prompt = next(p["prompt"] for p in prompts if p["name"] == "math")
ds_long = ds.with_options(timeout=3600, max_retries=0)  # длинный ответ: ждём до часа, без авто-повторов

path = UNLIMITED / MODEL / "math" / "run_1.md"
if path.exists():
    print(f"{path} уже есть, запрос не отправляем")
else:
    print(f"{datetime.now():%H:%M:%S} {MODEL}: запрос отправлен...")
    t0 = time.perf_counter()
    resp = ds_long.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": math_prompt}],
        temperature=TEMPERATURE,
    )
    latency = time.perf_counter() - t0
    u = resp.usage
    details = getattr(u, "completion_tokens_details", None)
    meta = {
        "model": MODEL,
        "task": "math",
        "run": 1,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "temperature": TEMPERATURE,
        "max_tokens": "none",
        "latency_s": round(latency, 3),
        "input_tokens": u.prompt_tokens,
        "output_tokens": u.completion_tokens,
        "reasoning_tokens": getattr(details, "reasoning_tokens", None) or 0,
        "total_tokens": u.total_tokens,
        "finish_reason": resp.choices[0].finish_reason,
        "cost_rub": round(u.total_tokens * PRICE_PER_1M[MODEL] / 1e6, 5),
    }
    header = "\n".join(f"{k}: {v}" for k, v in meta.items())
    path.parent.mkdir(parents=True, exist_ok=True)
    text = resp.choices[0].message.content
    path.write_text(f"---\n{header}\n---\n\n{text}\n", encoding="utf-8")
    pd.DataFrame([{**meta, "file": path.as_posix()}]).to_csv(UNL_CSV, mode="a", header=not UNL_CSV.exists(), index=False)
    print("    {latency_s} с, {output_tokens} ток. ответа (из них {reasoning_tokens} рассуждения), {finish_reason}".format(**meta))

07:30:11 deepseek-v4-pro: запрос отправлен...
    571.19 с, 4590 ток. ответа (из них 3361 рассуждения), stop


In [9]:
base = pd.read_csv(CSV_PATH)
base = base[(base["task"] == "math") & (base["model"] == MODEL)]
unl = pd.read_csv(UNL_CSV).iloc[-1]

comparison = pd.DataFrame({
    "с лимитом 8192 (среднее)": [base["latency_s"].mean(), base["output_tokens"].mean(), base["reasoning_tokens"].mean(),
                                 base["cost_rub"].mean(), (base["finish_reason"] == "length").sum()],
    "с лимитом 8192 (медиана)": [base["latency_s"].median(), base["output_tokens"].median(), base["reasoning_tokens"].median(),
                                 base["cost_rub"].median(), None],
    "без лимита": [unl["latency_s"], unl["output_tokens"], unl["reasoning_tokens"],
                   unl["cost_rub"], int(unl["finish_reason"] == "length")],
}, index=["время, с", "токены ответа", "из них рассуждения", "стоимость, ₽", "обрезано"]).round(3)
comparison.to_csv(UNLIMITED / "comparison.csv")
comparison

,с лимитом 8192 (среднее),с лимитом 8192 (медиана),без лимита
"время, с",403.804,417.057,571.190
токены ответа,8192.000,8192.000,4590.000
из них рассуждения,7484.800,7347.000,3361.000
"стоимость, ₽",0.154,0.154,0.088
обрезано,5.000,NaN,0.000
